In [86]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer


In [87]:
# Load the data
data = pd.read_csv('acs_conflict_combined.csv')

In [88]:
# Drop some unnecessary columns
data = data.drop(columns = ['ZCTA.1', 'month', 'admin2', 'Name', 'Name of ZIP Code', 'Rural_Urban_Commerical Status'])

In [89]:
# Recode urban variable to categorical
data['Rural_Urban_Commerical'] = data['Rural_Urban_Commerical'].astype('category')

categories = ['Rural', 'Urban', 'Commerical']  # Specify your categories if you know them

# Initialize the LabelEncoder
label_encoder = LabelEncoder()

# Fit the LabelEncoder to your categories
label_encoder.fit(categories)

# Transform your variable to encoded values
data['Rural_Urban_Commerical'] = label_encoder.transform(data['Rural_Urban_Commerical'])

In [90]:
# Count missing values
nan_count = data.isna().sum().sum()
print("Number of NaN values:", nan_count)

Number of NaN values: 4692


In [91]:
# Find and count values that indicate '-'
def count_dash(series):
    return series.eq('-').sum()

count_dash_data = data.applymap(lambda x: x == '-').sum().sum()

print("Number of values with '-' in the entire DataFrame:", count_dash_data)

C:\Users\danie\AppData\Local\Temp\ipykernel_88520\1411631116.py:5: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  count_dash_data = data.applymap(lambda x: x == '-').sum().sum()


Number of values with '-' in the entire DataFrame: 12768


In [92]:
# Find columns that hold '-'
columns_with_dash = data.apply(lambda x: '-' in x.values)

# Save those columns to list
columns_with_dash = columns_with_dash[columns_with_dash].index.tolist()

# It's median / mean columns that are calculated from other variables

In [93]:
data['Month'] = data['Month'].apply(lambda x: int(str(x)[-2:]))

In [94]:
# Replace each '-' with NaN, so can be imputed later
data.replace('-', np.nan, inplace=True)

In [95]:
# Looking for columns with the string '250,000+'
columns_with_string = data.applymap(lambda x: isinstance(x, str) and '250,000+' in x).any()

# Save those columns to a list
columns_containing_string = columns_with_string[columns_with_string].index.tolist()

# Inspect
print("Columns containing '250,000+':", columns_containing_string)

C:\Users\danie\AppData\Local\Temp\ipykernel_88520\1338546616.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  columns_with_string = data.applymap(lambda x: isinstance(x, str) and '250,000+' in x).any()


Columns containing '250,000+': ['Median Household Income, 2022', 'Median Family Income, 2022']


In [96]:
# Replace '250,000+' with '250,000' in the specified columns
for column in columns_containing_string:
    data[column] = data[column].str.replace('250,000+', '250,000')

In [97]:
# Select any column that is non-numeric
non_numeric_columns = data.select_dtypes(exclude=['number']).columns.tolist()

# Transform the non-numeric columns to numeric

for column in non_numeric_columns:
    data[column] = pd.to_numeric(data[column], errors='coerce')

In [84]:
# Imputing missing values

imputer = SimpleImputer(missing_values=np.nan, strategy='mean')

# Fit the imputer on the data and transform the column in one step
data= imputer.fit_transform(data)

In [98]:
# Save the final dataset to a CSV file
data.to_csv('acs_conflict_combined_cleaned.csv', index=False)